# Break AES-GCM By Reusing A Nonce, Then Make It Impossible

Companion notebook for the [Symmetric Encryption and Modes](https://codemia.io/courses/security_engineering/sec_symmetric_encryption_and_modes) lesson on Codemia.

**What you will build**: a hands on tour of the one rule authenticated encryption cannot survive breaking. With the
`cryptography` library you will recover a secret message from two ciphertexts that shared a nonce,
watch unauthenticated CTR mode accept a forged payment amount that GCM rejects, measure how fast
random nonces collide when the nonce space is small, reproduce the restart bug that makes counter
nonces repeat, and use associated data to stop ciphertexts being swapped between database rows.

## Setup

In [ ]:
%pip install -q cryptography numpy matplotlib

In [ ]:
import os
import math

import numpy as np
import matplotlib.pyplot as plt
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.exceptions import InvalidTag

key = AESGCM.generate_key(bit_length=256)
gcm = AESGCM(key)
rng = np.random.default_rng(3)

def xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

print('key bytes:', len(key), ' nonce bytes for GCM: 12')

## Concept recap

AES-GCM is counter mode plus an authenticator. Counter mode encrypts by XORing the plaintext with a
keystream that depends only on the key and the nonce, which makes it fast and parallel, and it means
two messages encrypted under the same key and nonce are XORed with the same keystream. The GHASH
authenticator is keyed by a value derived from the key, and a repeated nonce leaks enough to let an
attacker forge tags for that key.

So the contract is simple and absolute: never encrypt two messages with the same key and nonce. The
interesting engineering is in how systems break it without meaning to: random nonces in a space too
small for the volume, counters that reset on restart, two processes sharing a key and a counter.

## Two messages, one nonce

In [ ]:
nonce = os.urandom(12)
p1 = b'transfer 100 USD to account 4412 for invoice 22917'
p2 = b'new admin password for the billing service is Wt9q'
c1 = gcm.encrypt(nonce, p1, None)[:-16]    # drop the 16 byte tag, look at the ciphertext body
c2 = gcm.encrypt(nonce, p2, None)[:-16]

print('c1 xor c2 == p1 xor p2 :', xor(c1, c2) == xor(p1, p2))

# An attacker who knows or guesses one message recovers the other completely.
recovered = xor(xor(c1, c2), p1)
print('recovered second message:', recovered.decode())

The key never appeared, and AES was never attacked. The two ciphertexts cancel each other's keystream,
and what remains is the XOR of the plaintexts. Real messages are rarely secret in full: protocol
headers, JSON field names and padding are all predictable, and each predictable byte in one message
reveals the byte at the same position in the other. Crib dragging, sliding a guessed word along the
XOR and keeping positions that produce readable text, recovers the rest.

In [ ]:
# Crib dragging with no known plaintext at all: guess that the word 'password' appears somewhere
# in one of the two messages and see where it produces readable text in the other.
x = xor(c1, c2)
crib = b' password '
for pos in range(len(x) - len(crib) + 1):
    other = xor(x[pos:pos + len(crib)], crib)
    if all(32 <= b < 127 for b in other) and sum(chr(b).isalnum() or b == 32 for b in other) >= len(crib) - 1:
        print(f'offset {pos:>2}: other message reads {other!r}')

## Why the authenticator matters: a bit flip in CTR mode

In [ ]:
iv = os.urandom(16)
msg = b'{"to": "4412", "amount": 100}'
enc = Cipher(algorithms.AES(key), modes.CTR(iv)).encryptor()
ct = enc.update(msg) + enc.finalize()

# The attacker knows the format and wants 900 instead of 100. No key needed.
pos = msg.index(b'100')
tampered = bytearray(ct)
tampered[pos] ^= ord('1') ^ ord('9')
dec = Cipher(algorithms.AES(key), modes.CTR(iv)).decryptor()
print('CTR decrypts the forgery to:', (dec.update(bytes(tampered)) + dec.finalize()).decode())

nonce = os.urandom(12)
sealed = bytearray(gcm.encrypt(nonce, msg, None))
sealed[pos] ^= ord('1') ^ ord('9')
try:
    gcm.decrypt(nonce, bytes(sealed), None)
    print('GCM accepted the forgery')
except InvalidTag:
    print('GCM rejects the forgery: InvalidTag')

Encryption without authentication is malleable: flipping a ciphertext bit flips the same plaintext
bit, so anyone who knows the format can edit it. GCM's tag covers every ciphertext byte, and
decryption refuses to return anything if one is wrong. That is also why decrypting and then checking
the tag yourself is a bug: the library returns plaintext only after the tag passes, and code that
acts on unverified plaintext throws the protection away.

## How fast random nonces collide

In [ ]:
# The birthday bound: after n random nonces from a space of size N, the chance that some pair
# collided is about 1 - exp(-n^2 / 2N). Check it empirically in a deliberately tiny space.
def first_collision(bits):
    seen = set()
    space = 2 ** bits
    while True:
        v = int(rng.integers(space))
        if v in seen:
            return len(seen) + 1
        seen.add(v)

for bits in (16, 20, 24, 28):
    trials = [first_collision(bits) for _ in range(30)]
    print(f'{bits}-bit nonces: first collision after {np.median(trials):>8,.0f} messages '
          f'(sqrt of space = {2 ** (bits / 2):>8,.0f})')

In [ ]:
n = np.logspace(20, 40, 200, base=2)
p96 = -np.expm1(-n ** 2 / 2 ** 97)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.loglog(n, p96)
ax.axvline(2 ** 32, ls='--', color='gray', lw=0.8)
ax.text(2 ** 32 * 1.3, 1e-15, 'NIST limit for random nonces: 2^32 messages per key', fontsize=7)
ax.set_xlabel('messages encrypted under one key with random 96-bit nonces')
ax.set_ylabel('probability of any repeat')
ax.set_title('Random nonces are safe until they are not')
plt.tight_layout()
plt.show()
for e in (24, 32, 40, 48):
    print(f'2^{e} messages: p(repeat) = {-math.expm1(-(2 ** e) ** 2 / 2 ** 97):.2e}')

The first collision arrives near the square root of the space, every time. For 96-bit nonces that is
about two to the forty eight messages, which sounds like forever until you notice it is 280 trillion,
and the acceptable probability is far below even odds. Two to the thirty two messages keeps the risk
near one in eight billion, which is where key rotation limits such as the one in the CodeExecutor
come from. A service doing ten thousand encryptions a second reaches that in five days.

## The restart bug in counter nonces

In [ ]:
# Counter nonces never collide, as long as the counter never repeats. Two failure modes:
# a process that restarts and starts again at zero, and two processes sharing one key.
def counter_nonces(starts, per_run):
    out = []
    for s in starts:
        out.extend(range(s, s + per_run))
    return out

naive = counter_nonces([0, 0, 0], 1_000)              # three runs, counter kept in memory
persisted = counter_nonces([0, 1_000, 2_000], 1_000)  # counter saved and reloaded
two_proc = counter_nonces([0], 1_000) + counter_nonces([0], 1_000)
prefixed = [(1 << 64) | c for c in range(1_000)] + [(2 << 64) | c for c in range(1_000)]

for name, ns in [('in memory counter, 3 restarts', naive), ('persisted counter', persisted),
                 ('two processes, one counter each', two_proc), ('process id prefix + counter', prefixed)]:
    print(f'{name:<34} {len(ns):>5} nonces, {len(ns) - len(set(ns)):>5} repeats')

A counter in memory is a counter that restarts, and every restart replays the whole sequence under
the same key. Persisting it is fragile too, since a crash between encrypting and saving replays the
last value. The robust designs remove the shared state: give each process or each data key its own
nonce prefix, or derive a fresh key per process or per message, which is what envelope encryption in
the key management lesson does naturally. If none of that is possible, AES-GCM-SIV fails gracefully
on reuse, leaking only whether two messages were identical.

## Associated data binds a ciphertext to its context

In [ ]:
rows = {}
for user_id, ssn in [(17, b'123-45-6789'), (18, b'987-65-4321')]:
    n = os.urandom(12)
    rows[user_id] = (n, gcm.encrypt(n, ssn, f'users:{user_id}:ssn'.encode()))

# An attacker with write access to the table copies user 18's ciphertext into user 17's row.
rows[17] = rows[18]
for uid in (17, 18):
    n, ct = rows[uid]
    try:
        print(uid, gcm.decrypt(n, ct, f'users:{uid}:ssn'.encode()).decode())
    except InvalidTag:
        print(uid, 'InvalidTag: this ciphertext was not written for this row')

Without associated data, the swapped row decrypts cleanly and user 17 now reads user 18's number, and
the encryption never noticed. With the table, row and column in the associated data, the tag covers
where the value belongs as well as what it is. Associated data is not encrypted and not stored in the
ciphertext; the reader has to supply the same value, which is the point.

## Exercises

1. Extend the crib drag to try several cribs, such as ` the `, `account` and `USD`, and see how much
   of the second message you can reconstruct without being given the first.
2. Use the birthday formula to find how many messages per key a service can encrypt with random
   nonces before the repeat probability reaches one in a billion. At 50,000 encryptions a second,
   how often must it rotate keys?
3. Change the CTR forgery to turn `4412` into `9999`. Which bytes do you flip, and why does the same
   change still fail against GCM?
4. Repeat the associated data experiment, but put only the column name in the associated data. Which
   attack still works?

## What to take away

- Under one key, a nonce is used once, ever. A repeat leaks the XOR of the plaintexts and, in GCM,
  the ability to forge tags.
- Encryption without authentication can be edited by anyone who knows the format. Use an AEAD, and
  never act on plaintext before the tag is verified.
- Random 96-bit nonces collide near two to the forty eight messages; keep a key well below two to the
  thirty two encryptions and rotate.
- Counters fail on restart and when shared. Prefer per process prefixes, per process keys or envelope
  encryption, and AES-GCM-SIV where reuse cannot be ruled out.
- Put the context, table, row and field, in the associated data so ciphertexts cannot be moved.